# NLP Model Lab Exam: Dataset-Based Notebook

Run the notebook from top to bottom. Each section loads its own dataset. Change the CSV path and exact column names in that section's setup cell.

Install if needed:
```bash
pip install nltk spacy scikit-learn gensim transformers torch
python -m spacy download en_core_web_sm
```


## Shared imports and NLP resources

In [ ]:
import re, math, random
from collections import Counter

import numpy as np
import pandas as pd
import nltk
import spacy
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

nltk.download(["punkt", "punkt_tab", "averaged_perceptron_tagger",
               "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"], quiet=True)
nlp = spacy.load("en_core_web_sm")


## Section A: Load its dataset

In [ ]:
CSV_PATH = "data_A.csv"
TEXT_COL = "text"
LABEL_COL = "label"  # Use None if this dataset has no label column

df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column {TEXT_COL!r} not found. Available columns: {df.columns.tolist()}")
if LABEL_COL is not None and LABEL_COL not in df.columns:
    raise ValueError(f"Label column {LABEL_COL!r} not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
texts = df[TEXT_COL].tolist()

if LABEL_COL is not None:
    labeled_df = df.dropna(subset=[LABEL_COL]).copy()
    print("Labels:", labeled_df[LABEL_COL].unique().tolist())
else:
    labeled_df = pd.DataFrame()

print("Rows:", len(df))
display(df.head())

## Unit (a) -  Preprocessing, stemming and lemmatization

In [ ]:
def clean_tokens(raw_text):
    text = str(raw_text).lower()
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)

    contractions = {
        "don't": "do not", "can't": "cannot", "won't": "will not",
        "i'm": "i am", "it's": "it is", "isn't": "is not",
        "didn't": "did not", "you're": "you are", "we're": "we are"
    }
    for old, new in contractions.items():
        text = text.replace(old, new)

    text = text.replace(":)", " <happy> ").replace(":(", " <sad> ")
    return re.findall(r"<happy>|<sad>|[a-z]+", text)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    if tag.startswith("J"): return "a"
    if tag.startswith("V"): return "v"
    if tag.startswith("R"): return "r"
    return "n"

# Process every row in the dataset, not a fixed demonstration sentence.
preprocessed = []
for raw_text in texts:
    tokens = clean_tokens(raw_text)
    pos_tags = nltk.pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(w, wordnet_pos(tag)) for w, tag in pos_tags]
    preprocessed.append({"original": raw_text, "tokens": tokens,
                         "stems": [stemmer.stem(w) for w in tokens],
                         "lemmas": lemmas})

for item in preprocessed[:5]:
    print("\\nOriginal:", item["original"])
    print("Tokens:", item["tokens"])
    print("Stems :", item["stems"])
    print("Lemmas:", item["lemmas"])


## Unit (a) -  Bigram language model, smoothing, probability, perplexity and sampling

In [ ]:
sentences = []
for text in texts:
    words = clean_tokens(text)
    if words:
        sentences.append(["<s>"] + words + ["</s>"])

if not sentences:
    raise ValueError("No tokens found in TEXT_COL.")

unigrams = Counter(w for sent in sentences for w in sent)
bigrams = Counter()
context_counts = Counter()
for sent in sentences:
    for w1, w2 in zip(sent, sent[1:]):
        bigrams[(w1, w2)] += 1
        context_counts[w1] += 1

V = len(unigrams)
def bigram_prob(w1, w2):
    return (bigrams[(w1, w2)] + 1) / (context_counts[w1] + V)

# Evaluate the first actual dataset sentence.
test_sent = sentences[0]
log_prob = sum(math.log(bigram_prob(a, b)) for a, b in zip(test_sent, test_sent[1:]))
N = len(test_sent) - 1
print("Dataset sentence:", " ".join(test_sent[1:-1]))
print("Log probability:", log_prob)
print("Sentence probability:", math.exp(log_prob))
print("Perplexity:", math.exp(-log_prob / N))

# Sample a sequence from the dataset-trained bigram model.
random.seed(42)
vocab = [w for w in unigrams if w != "<s>"]
word, sampled = "<s>", []
for _ in range(20):
    candidates = [w for w in vocab if w != "<s>"]
    weights = [bigram_prob(word, w) for w in candidates]
    word = random.choices(candidates, weights=weights)[0]
    if word == "</s>": break
    sampled.append(word)
print("Sampled sentence:", " ".join(sampled))


## Section B: Load its dataset

In [ ]:
CSV_PATH = "data_B.csv"
TEXT_COL = "text"
LABEL_COL = "label"  # Use None if this dataset has no label column

df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column {TEXT_COL!r} not found. Available columns: {df.columns.tolist()}")
if LABEL_COL is not None and LABEL_COL not in df.columns:
    raise ValueError(f"Label column {LABEL_COL!r} not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
texts = df[TEXT_COL].tolist()

if LABEL_COL is not None:
    labeled_df = df.dropna(subset=[LABEL_COL]).copy()
    print("Labels:", labeled_df[LABEL_COL].unique().tolist())
else:
    labeled_df = pd.DataFrame()

print("Rows:", len(df))
display(df.head())

## Unit (b) -  POS tagging and open-class / closed-class words

In [ ]:
for raw_text in texts[:5]:
    tokens = nltk.word_tokenize(raw_text)
    tagged = nltk.pos_tag(tokens)
    open_class = [w for w, tag in tagged if tag.startswith(("NN", "VB", "JJ", "RB"))]
    closed_class = [w for w, tag in tagged if tag.startswith(("DT", "IN", "PRP", "CC"))]
    print("\\nText:", raw_text)
    print("POS tags:", tagged)
    print("Open-class words:", open_class)
    print("Closed-class words:", closed_class)


## Unit (b) -  CFG parsing and PP-attachment ambiguity (two trees)

In [ ]:
# CFG parsing requires a grammar that matches the vocabulary and structure.
# Try a dataset sentence first. If it does not match this tiny teaching grammar,
# parse the standard grammar example so the CFG/tree demonstration still works.
grammar = nltk.CFG.fromstring("""
S -> NP VP
VP -> V NP | VP PP
NP -> DT N | NP PP | 'i'
PP -> P NP
V -> 'saw'
DT -> 'the'
N -> 'man' | 'telescope'
P -> 'with'
""")
parser = nltk.ChartParser(grammar)

dataset_sentence = clean_tokens(texts[0])
trees = []
try:
    trees = list(parser.parse(dataset_sentence))
except ValueError:
    trees = []

if trees:
    sentence_used = dataset_sentence
    print("Parsing first dataset sentence:", " ".join(sentence_used))
else:
    sentence_used = "i saw the man with the telescope".split()
    trees = list(parser.parse(sentence_used))
    print("The first dataset sentence does not fit this small CFG.")
    print("Using grammar-compatible example:", " ".join(sentence_used))

print("Number of parse trees:", len(trees))
for tree in trees:
    tree.pretty_print()


## Unit (b) -  spaCy dependencies and aspect-opinion pairs

In [ ]:
# Dependency analysis and simple aspect-opinion extraction for each dataset row.
for raw_text in texts[:10]:
    doc = nlp(raw_text)
    print("\\nText:", raw_text)
    for token in doc:
        print(token.text, "--", token.dep_, "-->", token.head.text)

    pairs = []
    for token in doc:
        if token.dep_ == "amod":
            pairs.append((token.head.text, token.text))
        if token.dep_ == "acomp":
            subject = next((child.text for child in token.head.children
                            if child.dep_ == "nsubj"), None)
            if subject:
                pairs.append((subject, token.text))
    print("Aspect-opinion pairs:", pairs)


## Section C: Load its dataset

In [ ]:
CSV_PATH = "data_C.csv"
TEXT_COL = "text"
LABEL_COL = "label"  # Use None if this dataset has no label column

df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column {TEXT_COL!r} not found. Available columns: {df.columns.tolist()}")
if LABEL_COL is not None and LABEL_COL not in df.columns:
    raise ValueError(f"Label column {LABEL_COL!r} not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
texts = df[TEXT_COL].tolist()

if LABEL_COL is not None:
    labeled_df = df.dropna(subset=[LABEL_COL]).copy()
    print("Labels:", labeled_df[LABEL_COL].unique().tolist())
else:
    labeled_df = pd.DataFrame()

print("Rows:", len(df))
display(df.head())

## Unit (c) -  WordNet synsets, Wu-Palmer similarity and Lesk WSD

In [ ]:
context_text = texts[0]
context_tokens = nltk.word_tokenize(context_text)
candidate_words = [w.lower() for w in context_tokens if w.isalpha() and len(w) > 2]
word1 = candidate_words[0] if candidate_words else "bank"
word2 = candidate_words[1] if len(candidate_words) > 1 else "money"

syn1, syn2 = wn.synsets(word1), wn.synsets(word2)
print("Dataset context:", context_text)
print("Words:", word1, word2)
print("Synsets for first word:", syn1)
if syn1 and syn2:
    print("Wu-Palmer similarity (first senses):", syn1[0].wup_similarity(syn2[0]))

sense = lesk(context_tokens, word1)
print("Lesk sense:", sense)
print("Definition:", sense.definition() if sense else "No sense found")


## Unit (c) -  TF-IDF, cosine similarity and top-k retrieval

In [ ]:
tfidf = TfidfVectorizer()
doc_matrix = tfidf.fit_transform(texts)

QUERY_TEXT = None
query_text = QUERY_TEXT if QUERY_TEXT else texts[0]
query_vector = tfidf.transform([query_text])
scores = cosine_similarity(query_vector, doc_matrix).flatten()

k = min(3, len(texts))
top_k = scores.argsort()[::-1][:k]
print("Query:", query_text)
for i in top_k:
    print("Score:", round(float(scores[i]), 3), "| Row:", i, "| Text:", texts[i])


## Unit (c) -  PPMI with window ±2 and Word2Vec Skip-Gram

In [ ]:
tokenized = [clean_tokens(t) for t in texts]
tokenized = [s for s in tokenized if s]
if not tokenized:
    raise ValueError("No tokens available for PPMI/Word2Vec.")

word_counts = Counter(w for sent in tokenized for w in sent)
vocab = [w for w, count in word_counts.most_common(500)]
word_to_id = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))

for sent in tokenized:
    for i, word in enumerate(sent):
        if word not in word_to_id:
            continue
        for j in range(max(0, i - 2), min(len(sent), i + 3)):
            other = sent[j]
            if i != j and other in word_to_id:
                C[word_to_id[word], word_to_id[other]] += 1

row_sum = C.sum(axis=1, keepdims=True)
col_sum = C.sum(axis=0, keepdims=True)
ppmi = np.maximum(0, np.log2((C * C.sum() + 1e-9) / (row_sum @ col_sum + 1e-9)))
print("PPMI sample:\n", ppmi[:4, :4].round(2))

w2v = Word2Vec(sentences=tokenized, vector_size=50, window=2,
               min_count=1, sg=1, seed=42, workers=1)
if len(w2v.wv) > 1:
    print("Similar words:", w2v.wv.most_similar(vocab[0], topn=3))


## Section D: Load its dataset

In [ ]:
CSV_PATH = "data_D.csv"
TEXT_COL = "text"
LABEL_COL = "label"  # Use None if this dataset has no label column

df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column {TEXT_COL!r} not found. Available columns: {df.columns.tolist()}")
if LABEL_COL is not None and LABEL_COL not in df.columns:
    raise ValueError(f"Label column {LABEL_COL!r} not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
texts = df[TEXT_COL].tolist()

if LABEL_COL is not None:
    labeled_df = df.dropna(subset=[LABEL_COL]).copy()
    print("Labels:", labeled_df[LABEL_COL].unique().tolist())
else:
    labeled_df = pd.DataFrame()

print("Rows:", len(df))
display(df.head())

## Unit (d) -  Mentions, pair scoring and simple coreference chains

In [ ]:
for raw_text in texts[:5]:
    print("\nText:", raw_text)
    doc = nlp(raw_text)

    names = [e for e in doc.ents if e.label_ == "PERSON"]
    chunks = list(doc.noun_chunks)
    pronouns = [doc[t.i:t.i + 1] for t in doc if t.pos_ == "PRON"]
    print("Names:", [m.text for m in names])
    print("Noun chunks:", [m.text for m in chunks])
    print("Pronouns:", [m.text for m in pronouns])

    unique = {(m.start, m.end): m for m in names + chunks + pronouns}
    mentions = sorted(unique.values(), key=lambda m: m.start)

    male = {"mr", "john", "rahul", "he", "him", "his", "man", "boy"}
    female = {"mrs", "mary", "priya", "she", "her", "woman", "girl"}
    plural = {"they", "them", "their", "people"}
    pronoun_words = {"he", "him", "his", "she", "her", "they", "them", "their", "it", "its"}

    sentence_number = {}
    for i, sent in enumerate(doc.sents):
        for token in sent:
            sentence_number[token.i] = i

    def gender_number(m):
        words = set(re.findall(r"[a-z]+", m.text.lower()))
        gender = "M" if words & male else ("F" if words & female else "N")
        number = "P" if words & plural or m.root.tag_ in {"NNS", "NNPS"} else "S"
        return gender, number

    def pair_score(old, new):
        g1, n1 = gender_number(old)
        g2, n2 = gender_number(new)
        if n1 != n2 or (g1 != g2 and (g1 != "N" or g2 != "N")):
            return -10
        same_head = old.root.lemma_.lower() == new.root.lemma_.lower()
        is_pronoun = new.text.lower().strip(" .") in pronoun_words
        if not same_head and not is_pronoun:
            return -10
        score = 2
        if g1 == g2 and g1 != "N":
            score += 2
        if same_head:
            score += 2
        distance = sentence_number[new.root.i] - sentence_number[old.root.i]
        return score - distance

    chains = []
    mention_chain = []
    for i, mention in enumerate(mentions):
        best_j, best_score = -1, 0
        for j in range(i):
            score = pair_score(mentions[j], mention)
            if score > best_score:
                best_j, best_score = j, score
        if best_j == -1:
            chains.append([mention.text])
            mention_chain.append(len(chains) - 1)
        else:
            chain_id = mention_chain[best_j]
            chains[chain_id].append(mention.text)
            mention_chain.append(chain_id)

    print("Coreference chains:", [c for c in chains if len(c) > 1])


## Section E: Load its dataset

In [ ]:
CSV_PATH = "data_E.csv"
TEXT_COL = "text"
LABEL_COL = "label"  # Use None if this dataset has no label column

df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column {TEXT_COL!r} not found. Available columns: {df.columns.tolist()}")
if LABEL_COL is not None and LABEL_COL not in df.columns:
    raise ValueError(f"Label column {LABEL_COL!r} not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
texts = df[TEXT_COL].tolist()

if LABEL_COL is not None:
    labeled_df = df.dropna(subset=[LABEL_COL]).copy()
    print("Labels:", labeled_df[LABEL_COL].unique().tolist())
else:
    labeled_df = pd.DataFrame()

print("Rows:", len(df))
display(df.head())

## Unit (e) -  BERT fill-mask and sentiment pipelines

In [ ]:
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
words = texts[0].split()
if len(words) >= 2:
    masked_text = " ".join(words[:1] + ["[MASK]"] + words[2:])
else:
    masked_text = "NLP is [MASK]."
print("Fill-mask input:", masked_text)
print("Fill-mask:", fill_mask(masked_text, top_k=3))

sentiment = pipeline("sentiment-analysis")
sentiment_results = sentiment(texts[:min(10, len(texts))], truncation=True)
for raw_text, result in zip(texts[:10], sentiment_results):
    print("Text:", raw_text, "| Predicted sentiment:", result)


## Unit (e) -  Fine-tune BERT on dataset labels, accuracy, F1 and prediction\n\nRequires `TEXT_COL` and `LABEL_COL` to be set correctly in this section's setup cell. The model trains on dataset rows and evaluates on a held-out split.

In [ ]:
if LABEL_COL is None:
    raise ValueError("Set LABEL_COL in Cell 1 to fine-tune BERT on labeled data.")

bert_df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
bert_df[TEXT_COL] = bert_df[TEXT_COL].astype(str)
label_names = sorted(bert_df[LABEL_COL].astype(str).unique().tolist())
label_to_id = {name: i for i, name in enumerate(label_names)}
id_to_label = {i: name for name, i in label_to_id.items()}
bert_df["_label_id"] = bert_df[LABEL_COL].astype(str).map(label_to_id)

if len(label_names) < 2:
    raise ValueError("BERT classification needs at least two distinct labels.")
if len(bert_df) < 4:
    raise ValueError("Dataset is too small for a train/test split. Add more labeled rows.")

train_df, test_df = train_test_split(
    bert_df, test_size=0.2, random_state=42, stratify=bert_df["_label_id"]
)
train_texts = train_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()
train_labels = train_df["_label_id"].astype(int).tolist()
test_labels = test_df["_label_id"].astype(int).tolist()

torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(label_names)
)

inputs = tokenizer(train_texts, padding=True, truncation=True,
                   max_length=128, return_tensors="pt")
labels = torch.tensor(train_labels)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
EPOCHS = 3
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    loss = model(**inputs, labels=labels).loss
    loss.backward()
    optimizer.step()
    print("Epoch:", epoch + 1, "Loss:", round(loss.item(), 4))

# Evaluate on held-out test data (unlike the old hard-coded 9-sample example).
model.eval()
test_inputs = tokenizer(test_texts, padding=True, truncation=True,
                        max_length=128, return_tensors="pt")
with torch.no_grad():
    predictions = model(**test_inputs).logits.argmax(dim=1).numpy()

print("Label mapping:", label_to_id)
print("Test accuracy:", accuracy_score(test_labels, predictions))
print("Test F1 (weighted):", f1_score(test_labels, predictions, average="weighted", zero_division=0))

# Predict the label for a real row from the dataset.
new_text = test_texts[0]
new_inputs = tokenizer(new_text, return_tensors="pt", truncation=True, max_length=128)
with torch.no_grad():
    new_pred = model(**new_inputs).logits.argmax(dim=1).item()
print("Example dataset text:", new_text)
print("Actual label:", id_to_label[test_labels[0]])
print("Predicted label:", id_to_label[new_pred])
